# 260519 언어모델 파인튜닝 1

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w10_unstructured_docs/llm_260519_finetuning_1.ipynb)

In [ ]:
!pip install -q langchain langchain-community langchain-core langchain-openai openai

In [ ]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

client = OpenAI()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

## 강의 메모: 파인튜닝이 필요한 순간

- **시도 순서**: 프롬프트 엔지니어링 → RAG → 파인튜닝. 왼쪽일수록 싸고 빠름. 90%는 PE로 잡히면 나머지 10% 잡자고 FT하지 말 것.
- **"아는 문제 vs 행동 문제"**: 모델이 모르는 걸 보강할 땐 RAG, 이미 아는데 일관되게 행동을 못 할 땐 FT. 출력 포맷 흔들림·페르소나·말투·분류 일관성·시스템 프롬프트 비대화 → 전부 행동 문제 = FT 영역.
- **라디오 비유**: 모델은 모든 주파수를 받는 라디오, FT는 다이얼을 내 용도에 맞추는 것. "더 똑똑하게"가 아니라 "정해진 방식대로" 만드는 도구.
- **컨텍스트 비대화 함정**: 예외 잡으려 가이드 10개 붙이면 중간 지시는 모델이 까먹음(lost in the middle). 더 길게 쓸수록 오히려 안 듣는 악순환.
- **상반된 두 실패 모드**: ① 학습 부족(기존 지식이 강해 새 데이터 반영 X) ② 학습 과다 → **knowledge/language drift, catastrophic forgetting** (프랑스어 가르쳤더니 1+1을 못 푸는 격).
- **Full FT vs PEFT/LoRA**: 풀파인튜닝은 170B 파라미터 전부 갱신 → A100 80GB도 빠듯. LoRA는 W 동결, **W' = W + ΔW**의 ΔW만 작은 두 행렬(B×A, 0.1~1%)로 학습 → "책 통째 베끼기 vs 포스트잇 붙이기".
- **두 축의 조합**: FT는 "얼마나 바꿀까(Full/PEFT)" × "무엇으로 가르칠까(SFT/RLHF/DPO)". SFT=받아쓰기, DPO=선호쌍만 주면 됨(보상모델 불필요). 실무는 SFT로 큰 줄기 잡고 DPO로 매끈하게 다듬기.
- **OpenAI API의 FT**: 내부적으로 LoRA + SFT 자동 수행. JSONL 던지면 끝이지만 loss/eval 학습 지표를 볼 수 없는 게 단점 — 결과만 보고 판단해야 함.

## 강의 메모: 실무 팁

- **비용 구조 차이**: PE는 **운영 비용**(매 호출 토큰 누적), FT는 **일회성 큰 비용**(GPU·데이터). 호출량 많고 패턴 고정적이면 FT가 장기적으론 싸짐.
- **실패 비용은 PE가 압승**: 프롬프트는 다음 호출에 바로 고치면 끝, FT는 망친 모델 되돌릴 수 없고 재학습해야 함. 실험은 PE/RAG 충분히 한 뒤로 미룰 것.
- **분류 일관성은 짠 모델에서 더 티남**: 응답 생성은 비싼 모델, 분류·필터링은 mini 모델이 일반적이라 에러가 더 보임. "환불요청 vs 결제오류"처럼 사람도 헷갈리는 라벨이 FT 후보.
- **데이터 검증 4종**: 토큰 길이·필수 role·assistant 응답 누락·user 중복은 코드로 필터링. 단 동일 user에 **다른 assistant**가 더 위험(모델 혼란) — 단순 중복은 학습에 큰 해 없음.
- **데이터 운영**: 첫 학습보다 이후 운영이 더 귀찮음. 정반대 톤이 섞이면 드리프트 직행. 모델 버전 관리(`gpt-4o-mini-260519`, `_v2`) 습관 필요.
- **분할 & 평가**: 16개든 1만개든 train/val/test 8:1:1로 떼둘 것. 자동 메트릭은 실제 품질을 잘 못 반영 → **눈으로 보는 정성 평가** 병행.